# Tugas - Klasifikasi dengan Decision Tree

| Nama | NRP |
|------|-----|
| Isi Nama Kalian | Isi NRP Kalian |

> ⚠️ **Jangan lupa ganti Nama dan NRP pada tabel di atas sebelum dikumpulkan, dan rename file menjadi `DT_NRP.ipynb` sesuai NRP kalian.**

## Deskripsi Tugas
Pada tugas ini, kalian akan membangun model Decision Tree Classifier untuk menyelesaikan sebuah masalah klasifikasi.

Dataset: Taiwan Bankruptcy Prediction ([Kaggle - bankruptcy](https://www.kaggle.com/datasets/abbas829/bankruptcy), file lokal `bankruptcy.csv`).

Tujuan utama dari tugas ini adalah memahami cara kerja Decision Tree secara praktik, khususnya:
1. Perbedaan hasil antara kriteria split Gini Impurity dan Entropy
2. Pengaruh regularisasi (max_depth) terhadap gejala overfitting

Langkah-langkah yang harus dilakukan antara lain:
1. Persiapan Dataset dan Eksplorasi Awal
   - Memuat dataset, melihat struktur data, dan distribusi label.
2. Preprocessing
   - Memproses data agar siap digunakan dalam membangun model.
3. Eksperimen Model
   - Bangun model Decision Tree dengan kriteria Gini dan Entropy.
   - Lakukan eksperimen regularisasi dengan variasi nilai max_depth.
4. Evaluasi Model
   - Hitung metrik evaluasi dan visualisasikan hasilnya.
5. Analisis dan Kesimpulan
   - Bandingkan hasil eksperimen dan berikan kesimpulan.

# 1. Persiapan Dataset dan Eksplorasi Awal

In [1]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
RANDOM_STATE = 42

In [2]:
# Load dataset dan tampilkan informasi dasar
import os

# --- OPSI PATH: lokal vs Google Colab ---
# Secara default notebook memakai file lokal "bankruptcy.csv" (sudah ada di folder tugas).
# Kalau kamu running di Google Colab, isi COLAB_CSV_URL dengan link langsung ke CSV,
# contoh: link raw GitHub ("https://raw.githubusercontent.com/.../bankruptcy.csv")
# atau link download langsung Google Drive ("https://drive.google.com/uc?id=FILE_ID").
LOCAL_CSV = "bankruptcy.csv"
COLAB_CSV_URL = ""  # <-- isi link CSV di sini kalau pakai Colab, mis: "https://..."

if os.path.exists(LOCAL_CSV):
    CSV_PATH = LOCAL_CSV
    print(f"Memuat dataset lokal: {CSV_PATH}")
elif COLAB_CSV_URL:
    CSV_PATH = COLAB_CSV_URL
    print(f"File lokal tidak ditemukan, mengunduh CSV dari link:\n{CSV_PATH}")
else:
    raise FileNotFoundError(
        "File 'bankruptcy.csv' tidak ditemukan. "
        "Kalau di Colab: isi COLAB_CSV_URL di cell ini, "
        "atau upload bankruptcy.csv ke session Colab (atau mount Google Drive)."
    )

df = pd.read_csv(CSV_PATH)
# Bersihkan spasi berlebih pada nama kolom (nama kolom asli diawali spasi)
df.columns = df.columns.str.strip()

print("Jumlah baris dan kolom:", df.shape)
print("\nTipe data:")
print(df.dtypes.value_counts())
print("\nInfo kolom:")
df.info()
print("\nStatistik deskriptif (5 kolom pertama):")
display(df.describe().iloc[:, :5])
print("\nJumlah missing value per kolom (total):", int(df.isna().sum().sum()))
print("Jumlah baris duplikat:", int(df.duplicated().sum()))
print("\nJumlah unique per kolom (5 terkecil):")
display(df.nunique().sort_values().head(10))

FileNotFoundError: [Errno 2] No such file or directory: 'bankruptcy.csv'

In [ ]:
# Tampilkan distribusi kelas target dalam bentuk bar plot
TARGET = "Bankrupt?"
counts = df[TARGET].value_counts().sort_index()
props = df[TARGET].value_counts(normalize=True).sort_index()
print(counts)
print(props)
print(f"\nRasio imbalance: {counts[0]} : {counts[1]} (~{props[1]*100:.2f}% kelas bangkrut)")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x=TARGET, data=df, palette=["#4C78A8", "#F58518"], ax=ax)
ax.set_title("Distribusi Kelas Target (0=Tidak Bangkrut, 1=Bangkrut)")
ax.set_xlabel("Bankrupt?")
ax.set_ylabel("Jumlah")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width()/2., p.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Distribusi / hubungan antar fitur dibedakan berdasarkan kelas target
# Dataset punya 94 fitur numerik -> tampilkan histogram + boxplot untuk 6 fitur representatif
features_show = ["ROA(C) before interest and depreciation before interest",
                 "Operating Gross Margin",
                 "Borrowing dependency",
                 "Net worth/Assets",
                 "Current Ratio",
                 "Cash/Total Assets"]
for c in features_show:
    assert c in df.columns, f"kolom tidak ditemukan: {c}"

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, features_show):
    for cls, color in zip([0, 1], ["#4C78A8", "#F58518"]):
        vals = df.loc[df[TARGET] == cls, col].to_numpy()
        # clip ekstrim 1%-99% agar histogram terbaca (beberapa kolom ada outlier miliaran)
        lo, hi = np.percentile(df[col].to_numpy(), [1, 99])
        vals = np.clip(vals, lo, hi)
        ax.hist(vals, bins=40, label=f"kelas {cls}", color=color, alpha=0.5)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle("Distribusi 6 Fitur berdasarkan Kelas Target", y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, features_show):
    sns.boxplot(x=TARGET, y=col, data=df, palette=["#4C78A8", "#F58518"], ax=ax, showfliers=False)
    ax.set_title(col, fontsize=10)
plt.suptitle("Boxplot 6 Fitur per Kelas Target", y=1.02)
plt.tight_layout()
plt.show()

## Exploratory Data Analysis (EDA) tambahan

In [ ]:
# EDA tambahan: korelasi fitur terhadap target + cek kolom konstan/kategorikal
# 1. Semua fitur sudah numerik (tidak ada kolom object) -> tidak perlu encoding kategori
print("Kolom object:", df.select_dtypes(include="object").columns.tolist())

# 2. Cek kolom konstan (zero-variance) yang tidak berguna untuk split
const_cols = df.columns[df.nunique() <= 1].tolist()
print("Kolom konstan (1 nilai unik):", const_cols)

# 3. Korelasi Pearson tiap fitur terhadap target
corr = df.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(ascending=False)
print("\nTop 10 fitur berkorelasi positif dengan kebangkrutan:")
display(corr.head(10))
print("\nTop 10 fitur berkorelasi negatif dengan kebangkrutan:")
display(corr.tail(10))

plt.figure(figsize=(10, 6))
corr.head(15).sort_values().plot(kind="barh", color="#4C78A8")
plt.title("Top 15 Fitur dengan Korelasi Tertinggi terhadap Bankrupt?")
plt.xlabel("Korelasi Pearson")
plt.tight_layout()
plt.show()

# 4. Crosstab flag biner yang tersedia
if "Liability-Assets Flag" in df.columns:
    display(pd.crosstab(df["Liability-Assets Flag"], df[TARGET], normalize="index").round(3))

# 2. Preprocessing

In [ ]:
# Tangani missing value (jika ada)
print("Total missing:", int(df.isna().sum().sum()))
# Hasil: tidak ada missing value pada dataset ini, jadi tidak perlu imputasi.
# Jika ada missing, strategi yang aman dari leakage adalah imputasi memakai
# median/mean yang dihitung HANYA dari train set (mis. via SimpleImputer + Pipeline).
df_clean = df.copy()

In [ ]:
# Encoding fitur kategorikal (jika ada)
# Pengecekan: semua kolom bertipe numerik -> tidak ada yang perlu di-encoding.
print(df_clean.dtypes.value_counts())
# DecisionTreeClassifier sklearn butuh input angka; karena semua fitur sudah angka,
# langkah encoding dilewati. Satu-satunya kolom non-informatif adalah kolom konstan:
if "Net Income Flag" in df_clean.columns:
    print("Net Income Flag unique:", df_clean["Net Income Flag"].unique())
    df_clean = df_clean.drop(columns=["Net Income Flag"])
    print("Kolom 'Net Income Flag' dibuang (konstan=1, tidak bisa dipakai untuk split).")
print("Shape setelah drop kolom konstan:", df_clean.shape)

In [ ]:
# Pisahkan fitur (X) dan target (y), lalu split train/test 80:20 dengan stratify
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Distribusi train:", y_train.value_counts().to_dict())
print("Distribusi test :", y_test.value_counts().to_dict())
print(f"Proporsi kelas 1 - train: {y_train.mean():.4f}, test: {y_test.mean():.4f} (stratify menjaga proporsi)")

**Catatan penting:** Decision Tree tidak memerlukan feature scaling (StandardScaler/MinMaxScaler). Split pada Decision Tree dilakukan berdasarkan threshold per fitur satu per satu, bukan berdasarkan jarak antar titik seperti pada KNN atau SVM. Jadi kalian tidak perlu melakukan scaling pada tahap ini.

# 3. Eksperimen Model

## 3.1 Decision Tree dengan Kriteria Gini

Gini Impurity adalah kriteria default pada DecisionTreeClassifier di scikit-learn. Kriteria ini mengukur peluang kesalahan klasifikasi jika sebuah sampel dipilih secara acak dari suatu node.

In [ ]:
# Inisialisasi dan latih model DecisionTreeClassifier dengan criterion='gini'
model_gini = DecisionTreeClassifier(criterion="gini", random_state=RANDOM_STATE)
model_gini.fit(X_train, y_train)
print(f"Kedalaman pohon: {model_gini.get_depth()}, jumlah daun: {model_gini.get_n_leaves()}")
print(f"Akurasi train: {accuracy_score(y_train, model_gini.predict(X_train)):.4f}")

In [ ]:
# Prediksi pada test set (model Gini)
y_pred_gini = model_gini.predict(X_test)
print("Contoh 10 prediksi pertama:", y_pred_gini[:10])
print("Distribusi prediksi:", pd.Series(y_pred_gini).value_counts().to_dict())

## 3.2 Decision Tree dengan Kriteria Entropy

Entropy (Information Gain) adalah kriteria alternatif yang mengukur tingkat ketidakpastian distribusi kelas pada suatu node.

In [ ]:
# Latih model dengan criterion='entropy', hyperparameter lain sama agar adil
model_entropy = DecisionTreeClassifier(criterion="entropy", random_state=RANDOM_STATE)
model_entropy.fit(X_train, y_train)
print(f"Kedalaman pohon: {model_entropy.get_depth()}, jumlah daun: {model_entropy.get_n_leaves()}")
print(f"Akurasi train: {accuracy_score(y_train, model_entropy.predict(X_train)):.4f}")

In [ ]:
# Prediksi pada test set (model Entropy)
y_pred_entropy = model_entropy.predict(X_test)
print("Contoh 10 prediksi pertama:", y_pred_entropy[:10])
print("Distribusi prediksi:", pd.Series(y_pred_entropy).value_counts().to_dict())

## 3.3 Eksperimen Regularisasi: Pengaruh max_depth terhadap Overfitting

Decision Tree yang dibiarkan tumbuh tanpa batas cenderung overfitting terhadap data training. Pada bagian ini kalian akan mengamati bagaimana perubahan nilai max_depth memengaruhi akurasi training dan testing.

In [ ]:
# Latih beberapa model dengan variasi max_depth (kriteria terbaik sementara: 'gini')
depths = [1, 2, 3, 5, 10, None]
models_depth = {}
for d in depths:
    clf = DecisionTreeClassifier(criterion="gini", max_depth=d, random_state=RANDOM_STATE)
    clf.fit(X_train, y_train)
    models_depth[d] = clf
    print(f"max_depth={str(d):>4} -> depth aktual={clf.get_depth():>2}, leaves={clf.get_n_leaves():>3}, "
          f"train_acc={accuracy_score(y_train, clf.predict(X_train)):.4f}")

In [ ]:
# Hitung akurasi train & test untuk setiap max_depth
train_acc, test_acc = [], []
for d in depths:
    clf = models_depth[d]
    train_acc.append(accuracy_score(y_train, clf.predict(X_train)))
    test_acc.append(accuracy_score(y_test, clf.predict(X_test)))

results_depth = pd.DataFrame({
    "max_depth": [str(d) for d in depths],
    "train_accuracy": train_acc,
    "test_accuracy": test_acc,
    "gap(train-test)": np.array(train_acc) - np.array(test_acc),
})
display(results_depth)

In [ ]:
# Plot akurasi training vs testing terhadap max_depth
x = np.arange(len(depths))
labels = [str(d) for d in depths]
plt.figure(figsize=(8, 5))
plt.plot(x, train_acc, marker="o", label="Train accuracy")
plt.plot(x, test_acc, marker="s", label="Test accuracy")
plt.xticks(x, labels)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Akurasi Training vs Testing untuk Berbagai max_depth (criterion=gini)")
plt.legend()
plt.grid(True, alpha=0.3)
for i, (tr, te) in enumerate(zip(train_acc, test_acc)):
    plt.text(i, tr + 0.001, f"{tr:.3f}", ha="center", fontsize=8)
    plt.text(i, te - 0.004, f"{te:.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.show()

# 4. Evaluasi Model

In [ ]:
# Metrik evaluasi (Accuracy, Precision, Recall, F1) untuk Gini vs Entropy
def scores(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
    }

comparison = pd.DataFrame([scores(y_test, y_pred_gini), scores(y_test, y_pred_entropy)],
                          index=["Gini", "Entropy"]).round(4)
display(comparison)
print("\nClassification report - Gini:")
print(classification_report(y_test, y_pred_gini, digits=4, zero_division=0))
print("Classification report - Entropy:")
print(classification_report(y_test, y_pred_entropy, digits=4, zero_division=0))

In [ ]:
# Confusion Matrix Gini vs Entropy dalam satu baris subplot
cm_gini = confusion_matrix(y_test, y_pred_gini)
cm_ent = confusion_matrix(y_test, y_pred_entropy)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, cm, title in zip(axes, [cm_gini, cm_ent], ["Confusion Matrix - Gini", "Confusion Matrix - Entropy"]):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
    ax.set_title(title)
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi struktur Decision Tree terbaik (Gini unggul tipis -> visualkan, batasi depth agar terbaca)
best_model = model_gini  # ganti ke model_entropy jika entropy lebih baik pada run kalian
plt.figure(figsize=(20, 8))
plot_tree(best_model, max_depth=3, feature_names=list(X.columns),
          class_names=["Tidak Bangkrut", "Bangkrut"],
          filled=True, rounded=True, fontsize=8, impurity=True)
plt.title("Struktur Decision Tree (criterion=gini, ditampilkan hingga kedalaman 3)", fontsize=14)
plt.show()

In [ ]:
# Feature importance model terbaik dalam bar chart horizontal (Top 15)
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=True)
top15 = importances.tail(15)
print(top15.tail(5).round(4))

plt.figure(figsize=(10, 6))
top15.plot(kind="barh", color="#4C78A8")
plt.title("Top 15 Feature Importance (Decision Tree - Gini)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

# 5. Analisis

### Analisis hasil eksperimen

**1. Gini vs Entropy — apakah berbeda signifikan? Mengapa?**
- Pada run dengan `random_state=42`, split 80:20 stratified: model **Gini** meraih akurasi test **~0.9611** (presisi ~0.395, recall ~0.386, F1 ~0.391), sedangkan **Entropy** meraih akurasi test **~0.9589** (presisi ~0.357, recall ~0.341, F1 ~0.349). Selisihnya kecil (<0.5% akurasi, ~0.04 F1) sehingga **tidak signifikan secara praktis**.
- Hal ini wajar: Gini dan Entropy adalah dua ukuran impurity yang berbeda rumus tetapi monotonik mirip — keduanya memilih split yang memurnikan node. Pada dataset dengan banyak fitur kontinu yang berkorelasi (mis. ROA, Net Value Per Share), urutan split terbaik yang ditemukan keduanya hampir sama; pohon yang dihasilkan memang mirip (kedalaman ~18–20, ~100–124 daun tanpa batas).
- Perbedaan kecil muncul karena Entropy (logaritmik) sedikit lebih sensitif terhadap distribusi probabilitas kelas, sedangkan Gini (kuadratik) sedikit lebih cepat dihitung. Pada data sangat imbalance seperti ini (3.2% bangkrut), keduanya sama-sama kesulitan menaikkan recall kelas minoritas tanpa mengorbankan presisi.
- Catatan penting: **akurasi ~96% menipu** — model naif yang selalu memprediksi "tidak bangkrut" pun mendapat ~96.8%. Metrik yang jujur adalah presisi/recall/F1 kelas bangkrut yang masih rendah (~0.35–0.46), terlihat dari confusion matrix (±27 false negative/false positive dari 44 kasus bangkrut di test set).

**2. Eksperimen regularisasi max_depth — kapan overfitting muncul?**
- `max_depth=1`: train acc = test acc ≈ 0.9677, F1 = 0.00 — pohon hanya 1 split, **underfitting** (memprediksi hampir semua sebagai kelas mayoritas).
- `max_depth=2–3`: test acc mencapai puncak (**~0.9699–0.9714** untuk Gini) dan F1 mulai naik (~0.16–0.32). Ini daerah **kompromi terbaik bias-variance**.
- `max_depth≥5`: train acc terus naik (0.977 → 0.995 → 1.000 pada depth 10/None) sementara test acc **stagnan lalu turun** (0.969 → 0.961 → 0.961). Gap train−test melebar dari ~0.008 menjadi ~0.039 — inilah **ciri klasik overfitting**: pohon menghafal noise/outlier training (daun Lonjakan 21 → 88 → 124) tetapi generalisasi memburuk.
- Kesimpulan: gejala overfitting mulai terlihat jelas pada **max_depth > 3–5**. Titik sweet-spot pada eksperimen ini adalah **max_depth=3** (test acc tertinggi, gap terkecil).

**3. Fitur paling berpengaruh (feature importance model Gini):**
- Peringkat 1 jauh di atas yang lain: **Borrowing dependency** (~0.21) — ketergantungan pada pinjaman adalah sinyal dominan kebangkrutan.
- Berikutnya: **Continuous interest rate (after tax)** (~0.08), **Persistent EPS in the Last Four Seasons** (~0.06), **Net Value Growth Rate** (~0.06), **Quick Ratio** (~0.04), lalu Operating Expense Rate, Interest-bearing debt interest rate, Total Asset Growth Rate, dsb.
- Secara bisnis ini masuk akal: kombinasi leverage/struktur utang + profitabilitas (EPS, ROA) + likuiditas (Quick Ratio, Cash/Total Assets) yang menjadi pemisah utama di node-node atas pohon (terlihat pada visualisasi `plot_tree`).


# 6. Kesimpulan dan Saran

### Kesimpulan
1. Model Decision Tree berhasil dilatih pada dataset bankruptcy (6819 baris, 94 fitur numerik setelah membuang kolom konstan `Net Income Flag`, tanpa missing value, split 80:20 stratified). Pohon tanpa batas mencapai akurasi test ~96% tetapi F1 kelas bangkrut hanya ~0.35–0.39 karena data sangat imbalance (3.2%).
2. **Gini sedikit lebih baik dari Entropy** pada eksperimen ini ( naming: test acc 0.9611 vs 0.9589; F1 0.39 vs 0.35), namun perbedaannya tidak signifikan karena kedua kriteria memilih split yang hampir identik.
3. **Regularisasi max_depth terbukti mengendalikan overfitting**: pohon tak terbatas menghafal training (train acc 1.000, >100 daun) sedangkan `max_depth=3` memberi generalisasi terbaik (test acc ~0.971, gap train–test minimal). Overfitting muncul pada `max_depth > 3–5` dengan ciri train acc naik terus dan test acc menurun/stagnan.
4. Fitur terpenting adalah **Borrowing dependency**, diikuti Continuous interest rate (after tax), Persistent EPS, Net Value Growth Rate, dan Quick Ratio — sejalan dengan intuisi keuangan bahwa leverage dan profitabilitas menentukan kebangkrutan.

### Saran
- Tangani imbalance secara eksplisit: coba `class_weight="balanced"`, SMOTE (hanya pada train set), atau tuning threshold probabilitas untuk menaikkan recall kelas bangkrut; evaluasi dengan PR-AUC / F1 bukan akurasi saja.
- Coba **cost-complexity pruning (`ccp_alpha`)** dan `min_samples_leaf` / `min_samples_split` sebagai regularisasi alternatif selain max_depth; gunakan **stratified cross-validation + GridSearchCV** agar pemilihan hyperparameter lebih stabil.
- Bandingkan dengan ensemble berbasis tree (**Random Forest, Gradient Boosting/XGBoost**) yang biasanya jauh lebih kuat pada data tabular imbalance, serta analisis kesalahan (error analysis) pada false negative — karena melewatkan perusahaan bangkrut biayanya paling mahal.
